# Enterprise SaaS Churn Capstone: Adoption & Value Realization
Required Capstone Assignment 20.1 — Initial Report and Exploratory Data Analysis

Synthetic teaching data only: 50 accounts, five churn-positive labels.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, average_precision_score, roc_auc_score, ConfusionMatrixDisplay, PrecisionRecallDisplay
RANDOM_STATE=42
df=pd.read_csv('../data/saas_telemetry.csv')
print('Shape:',df.shape); display(df.head()); display(df.isna().sum().to_frame('missing_count'))
display(df['churn'].value_counts().rename('accounts'))


Shape: (50, 16)


churn
0    45
1     5
Name: accounts, dtype: int64

## Data quality and cleaning
Check duplicate account identifiers and missingness. Flag numeric outliers using 1.5×IQR; do not automatically delete plausible enterprise-scale values.

In [2]:
duplicate_count=int(df['account_id'].duplicated().sum())
df=df.drop_duplicates(subset=['account_id']).copy()
print('Duplicate IDs removed:',duplicate_count)
display(df.isna().sum().to_frame('missing_count'))
outlier_rows=[]
for col in df.select_dtypes(include='number').columns.drop('churn'):
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    outlier_rows.append({'feature':col,'outlier_count':int(((df[col]<lo)|(df[col]>hi)).sum()),'lower':lo,'upper':hi})
display(pd.DataFrame(outlier_rows))


Duplicate IDs removed: 0


missing_count
csat_score                                  2
api_calls_monthly                           1
executive_sponsor_qbr_attendance             1

                              feature  outlier_count
0             contract_tenure_months              0
1          annual_recurring_revenue              7
2                     seat_licenses              7
3            license_utilization_pct              0
4                       dau_mau_ratio              0
5                 core_modules_active              0
6                  api_calls_monthly              4
7       kpi_milestones_achieved_pct              0
8               automated_workflows              0
9  executive_sponsor_qbr_attendance              0
10                       csat_score              0

## Feature engineering and EDA
Derived features normalize commercial value and quantify adoption/value gaps. Group comparisons are descriptive, not causal.

In [3]:
df['arr_per_seat']=df['annual_recurring_revenue']/df['seat_licenses'].replace(0,np.nan)
df['adoption_gap']=1-df['license_utilization_pct']
df['value_deficit']=1-df['kpi_milestones_achieved_pct']
fig,axes=plt.subplots(2,3,figsize=(15,9))
df['churn'].value_counts().sort_index().plot(kind='bar',ax=axes[0,0],title='Churn class distribution')
df.groupby('tier')['churn'].mean().plot(kind='bar',ax=axes[0,1],title='Churn rate by tier')
df['license_utilization_pct'].plot(kind='hist',bins=10,ax=axes[0,2],title='License utilization')
corr=df.select_dtypes(include='number').corr(); im=axes[1,0].imshow(corr,vmin=-1,vmax=1,aspect='auto'); axes[1,0].set_title('Numeric correlation'); axes[1,0].set_xticks(range(len(corr)),corr.columns,rotation=90); axes[1,0].set_yticks(range(len(corr)),corr.columns); fig.colorbar(im,ax=axes[1,0])
df.boxplot(column='csat_score',by='churn',ax=axes[1,1]); axes[1,1].set_title('CSAT by churn')
df.boxplot(column='kpi_milestones_achieved_pct',by='churn',ax=axes[1,2]); axes[1,2].set_title('KPI milestones by churn')
fig.suptitle(''); plt.tight_layout(); plt.show()
display(df.groupby('churn')[['license_utilization_pct','dau_mau_ratio','core_modules_active','kpi_milestones_achieved_pct','csat_score']].mean().T)


Model rows: 50 | churn prevalence: 0.1


                                  0         1
license_utilization_pct      0.661511  0.508000
dau_mau_ratio                0.524156  0.422800
core_modules_active          4.888889  3.800000
kpi_milestones_achieved_pct   0.694778  0.484800
csat_score                   3.302326  2.600000

## Baseline Logistic Regression
Numeric and categorical preprocessing is fitted only on the training partition. Recall and Average Precision are emphasized because churn is the minority class.

In [4]:
target,id_col='churn','account_id'; X=df.drop(columns=[target,id_col]); y=df[target].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=RANDOM_STATE,stratify=y)
numeric=X.select_dtypes(include='number').columns.tolist(); categorical=X.select_dtypes(exclude='number').columns.tolist()
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
prep=ColumnTransformer([('num',num_pipe,numeric),('cat',cat_pipe,categorical)])
model=Pipeline([('prep',prep),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))])
model.fit(X_train,y_train); pred=model.predict(X_test); score=model.predict_proba(X_test)[:,1]
metrics={'Recall':recall_score(y_test,pred,zero_division=0),'Precision':precision_score(y_test,pred,zero_division=0),'PR-AUC':average_precision_score(y_test,score),'ROC-AUC':roc_auc_score(y_test,score) if y_test.nunique()==2 else np.nan}
display(pd.Series(metrics).to_frame('test_score')); display(y_test.value_counts().rename('test_accounts'))
fig,axes=plt.subplots(1,2,figsize=(10,4)); ConfusionMatrixDisplay.from_predictions(y_test,pred,ax=axes[0]); PrecisionRecallDisplay.from_predictions(y_test,score,ax=axes[1]); plt.tight_layout(); plt.show()


                            test_score
Recall                        0.000000
Precision                     0.000000
PR-AUC                        0.200000
ROC-AUC                       0.666667

churn
0    12
1     1
Name: test_accounts, dtype: int64

Confusion matrix: 12 retained correctly classified; 1 churn case missed; 0 churn cases detected.


## Interpretation and limitations
Recall is the fraction of churn cases detected; precision is the fraction of flagged accounts that churned; Average Precision summarizes the precision-recall trade-off. With only five positive labels overall and one in the test split, the metrics are unstable. This is a coursework demonstration, not a validated churn model. Production use requires larger governed data, time-based validation, leakage controls, calibration, segment evaluation and human review.